In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import files
uploaded = files.upload()   # select fraudTrain.csv and fraudTest.csv when prompted

In [ ]:
train_df = pd.read_csv('fraudTrain.csv')
test_df = pd.read_csv('fraudTest.csv')

print(train_df.shape)
print(test_df.shape)

(1296675, 23)
(555719, 23)


In [ ]:
train_df.head()

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,...,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,...,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,...,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,...,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,...,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0


In [ ]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1296675 entries, 0 to 1296674
Data columns (total 23 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   Unnamed: 0             1296675 non-null  int64  
 1   trans_date_trans_time  1296675 non-null  object 
 2   cc_num                 1296675 non-null  int64  
 3   merchant               1296675 non-null  object 
 4   category               1296675 non-null  object 
 5   amt                    1296675 non-null  float64
 6   first                  1296675 non-null  object 
 7   last                   1296675 non-null  object 
 8   gender                 1296675 non-null  object 
 9   street                 1296675 non-null  object 
 10  city                   1296675 non-null  object 
 11  state                  1296675 non-null  object 
 12  zip                    1296675 non-null  int64  
 13  lat                    1296675 non-null  float64
 14  long              

In [ ]:
train_df['is_fraud'].value_counts()

,count
is_fraud,
0,1289169
1,7506


In [ ]:
train_df['is_fraud'].value_counts(normalize=True) * 100   # as percentages

,proportion
is_fraud,
0,99.421135
1,0.578865


In [ ]:
train_df.isnull().sum()

,0
Unnamed: 0,0
trans_date_trans_time,0
cc_num,0
merchant,0
category,0
amt,0
first,0
last,0
gender,0
street,0


In [ ]:
train_df.columns.tolist()

['Unnamed: 0',
 'trans_date_trans_time',
 'cc_num',
 'merchant',
 'category',
 'amt',
 'first',
 'last',
 'gender',
 'street',
 'city',
 'state',
 'zip',
 'lat',
 'long',
 'city_pop',
 'job',
 'dob',
 'trans_num',
 'unix_time',
 'merch_lat',
 'merch_long',
 'is_fraud']

In [ ]:
train_df.groupby('is_fraud')['amt'].describe()

,count,mean,std,min,25%,50%,75%,max
is_fraud,,,,,,,,
0,1289169.0,67.667110,154.007971,1.00,9.6100,47.280,82.540,28948.90
1,7506.0,531.320092,390.560070,1.06,245.6625,396.505,900.875,1376.04


In [ ]:
train_df.groupby('category')['is_fraud'].mean().sort_values(ascending=False)

,is_fraud
category,
shopping_net,0.017561
misc_net,0.014458
grocery_pos,0.014098
shopping_pos,0.007225
gas_transport,0.004694
misc_pos,0.003139
grocery_net,0.002948
travel,0.002864
entertainment,0.002478


In [ ]:
train_df['trans_date_trans_time'] = pd.to_datetime(train_df['trans_date_trans_time'])
train_df['hour'] = train_df['trans_date_trans_time'].dt.hour
train_df.groupby('hour')['is_fraud'].mean()

,is_fraud
hour,
0,0.014940
1,0.015349
2,0.014652
3,0.014239
4,0.001099
5,0.001423
6,0.000946
7,0.001327
8,0.001153


In [ ]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

train_df['distance_km'] = train_df.apply(
    lambda row: haversine(row['lat'], row['long'], row['merch_lat'], row['merch_long']), axis=1
)

train_df.groupby('is_fraud')['distance_km'].describe()

,count,mean,std,min,25%,50%,75%,max
is_fraud,,,,,,,,
0,1289169.0,76.113756,29.119051,0.022255,55.332701,78.233012,98.504498,152.117173
1,7506.0,76.268330,28.752602,0.738769,55.632890,77.931954,98.391090,144.522410


In [ ]:
train_df['dob'] = pd.to_datetime(train_df['dob'])
train_df['age'] = 2020 - train_df['dob'].dt.year   # dataset is from ~2019-2020
train_df.groupby('is_fraud')['age'].describe()

,count,mean,std,min,25%,50%,75%,max
is_fraud,,,,,,,,
0,1289169.0,46.726131,17.368274,15.0,33.0,45.0,58.0,96.0
1,7506.0,49.561684,18.841440,15.0,34.0,48.0,62.0,95.0


In [ ]:
train_df.groupby('gender')['is_fraud'].mean()

,is_fraud
gender,
F,0.005262
M,0.006426


In [ ]:
train_df.groupby('is_fraud')['city_pop'].describe()

,count,mean,std,min,25%,50%,75%,max
is_fraud,,,,,,,,
0,1289169.0,88775.228137,301806.545204,23.0,743.0,2456.0,20328.0,2906700.0
1,7506.0,97276.763256,326581.466670,23.0,746.5,2623.0,21437.0,2906700.0


In [ ]:
train_df['is_night'] = train_df['hour'].isin([22, 23, 0, 1, 2, 3]).astype(int)
train_df.groupby('is_night')['is_fraud'].mean()

,is_fraud
is_night,
0,0.001153
1,0.020867


In [ ]:
train_df['dob'] = pd.to_datetime(train_df['dob'])
train_df['age'] = 2020 - train_df['dob'].dt.year

features = ['amt', 'category', 'is_night', 'age', 'gender', 'city_pop']
target = 'is_fraud'

X = train_df[features].copy()
y = train_df[target].copy()

X.head()

,amt,category,is_night,age,gender,city_pop
0,4.97,misc_net,1,32,F,3495
1,107.23,grocery_pos,1,42,F,149
2,220.11,entertainment,1,58,M,4154
3,45.00,gas_transport,1,53,M,1939
4,41.96,misc_pos,1,34,M,99


In [ ]:
X = pd.get_dummies(X, columns=['category', 'gender'], drop_first=True)
X.head()

,amt,is_night,age,city_pop,category_food_dining,category_gas_transport,category_grocery_net,category_grocery_pos,category_health_fitness,category_home,category_kids_pets,category_misc_net,category_misc_pos,category_personal_care,category_shopping_net,category_shopping_pos,category_travel,gender_M
0,4.97,1,32,3495,False,False,False,False,False,False,False,True,False,False,False,False,False,False
1,107.23,1,42,149,False,False,False,True,False,False,False,False,False,False,False,False,False,False
2,220.11,1,58,4154,False,False,False,False,False,False,False,False,False,False,False,False,False,True
3,45.00,1,53,1939,False,True,False,False,False,False,False,False,False,False,False,False,False,True
4,41.96,1,34,99,False,False,False,False,False,False,False,False,True,False,False,False,False,True


In [ ]:
print(X.shape)
X.columns.tolist()

(1296675, 18)


['amt',
 'is_night',
 'age',
 'city_pop',
 'category_food_dining',
 'category_gas_transport',
 'category_grocery_net',
 'category_grocery_pos',
 'category_health_fitness',
 'category_home',
 'category_kids_pets',
 'category_misc_net',
 'category_misc_pos',
 'category_personal_care',
 'category_shopping_net',
 'category_shopping_pos',
 'category_travel',
 'gender_M']

In [ ]:
test_df['trans_date_trans_time'] = pd.to_datetime(test_df['trans_date_trans_time'])
test_df['hour'] = test_df['trans_date_trans_time'].dt.hour
test_df['is_night'] = test_df['hour'].isin([22, 23, 0, 1, 2, 3]).astype(int)

test_df['dob'] = pd.to_datetime(test_df['dob'])
test_df['age'] = 2020 - test_df['dob'].dt.year

X_test = test_df[features].copy()
y_test = test_df[target].copy()

X_test = pd.get_dummies(X_test, columns=['category', 'gender'], drop_first=True)

# align columns in case test set is missing a category train_df had (or vice versa)
X_test = X_test.reindex(columns=X.columns, fill_value=0)

print(X_test.shape)

(555719, 18)


In [ ]:
from sklearn.preprocessing import StandardScaler

numeric_cols = ['amt', 'age', 'city_pop']

scaler = StandardScaler()
X[numeric_cols] = scaler.fit_transform(X[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])  # transform only, not fit — reuse train's scaler

X.head()

,amt,is_night,age,city_pop,category_food_dining,category_gas_transport,category_grocery_net,category_grocery_pos,category_health_fitness,category_home,category_kids_pets,category_misc_net,category_misc_pos,category_personal_care,category_shopping_net,category_shopping_pos,category_travel,gender_M
0,-0.407826,1,-0.848322,-0.282589,False,False,False,False,False,False,False,True,False,False,False,False,False,False
1,0.230039,1,-0.272898,-0.293670,False,False,False,True,False,False,False,False,False,False,False,False,False,False
2,0.934149,1,0.647781,-0.280406,False,False,False,False,False,False,False,False,False,False,False,False,False,True
3,-0.158132,1,0.360069,-0.287742,False,True,False,False,False,False,False,False,False,False,False,False,False,True
4,-0.177094,1,-0.733237,-0.293835,False,False,False,False,False,False,False,False,True,False,False,False,False,True


In [ ]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
log_reg.fit(X, y)

print("Logistic Regression trained.")

Logistic Regression trained.


In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(class_weight='balanced', max_depth=10, random_state=42)
dt.fit(X, y)

print("Decision Tree trained.")

Decision Tree trained.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(class_weight='balanced', n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X, y)

print("Random Forest trained.")

Random Forest trained.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_pred_lr = log_reg.predict(X_test)

print("=== Logistic Regression ===")
print(classification_report(y_test, y_pred_lr))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))
print("ROC-AUC:", roc_auc_score(y_test, log_reg.predict_proba(X_test)[:, 1]))

=== Logistic Regression ===
              precision    recall  f1-score   support

           0       1.00      0.86      0.93    553574
           1       0.02      0.89      0.05      2145

    accuracy                           0.86    555719
   macro avg       0.51      0.88      0.49    555719
weighted avg       1.00      0.86      0.92    555719

Confusion Matrix:
[[476978  76596]
 [   231   1914]]
ROC-AUC: 0.9471582525867951


In [ ]:
y_pred_dt = dt.predict(X_test)

print("=== Decision Tree ===")
print(classification_report(y_test, y_pred_dt))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_dt))
print("ROC-AUC:", roc_auc_score(y_test, dt.predict_proba(X_test)[:, 1]))

=== Decision Tree ===
              precision    recall  f1-score   support

           0       1.00      0.96      0.98    553574
           1       0.09      0.97      0.16      2145

    accuracy                           0.96    555719
   macro avg       0.54      0.97      0.57    555719
weighted avg       1.00      0.96      0.98    555719

Confusion Matrix:
[[531404  22170]
 [    59   2086]]
ROC-AUC: 0.9859563381578504


In [ ]:
y_pred_rf = rf.predict(X_test)

print("=== Random Forest ===")
print(classification_report(y_test, y_pred_rf))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1]))

=== Random Forest ===
              precision    recall  f1-score   support

           0       1.00      0.98      0.99    553574
           1       0.13      0.95      0.23      2145

    accuracy                           0.98    555719
   macro avg       0.56      0.96      0.61    555719
weighted avg       1.00      0.98      0.98    555719

Confusion Matrix:
[[539809  13765]
 [   110   2035]]
ROC-AUC: 0.9919997695332158
